In [1]:
!nvidia-smi

Mon Aug 24 03:17:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   33C    P0             41W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# a folder to keep everything we want to survive between sessions
!mkdir -p /content/drive/MyDrive/silent_speech

In [4]:
%cd /content
!git clone https://github.com/MatteoFasulo/silent_speech.git
%cd /content/silent_speech

# only the alignment submodule is needed for the text task
!git submodule update --init text_alignments
# unpack the TextGrid alignments into ./text_alignments/<session>/
!tar -xzf text_alignments/text_alignments.tar.gz

/content
Cloning into 'silent_speech'...
remote: Enumerating objects: 489, done.
remote: Counting objects: 100% (333/333), done.
remote: Compressing objects: 100% (194/194), done.
remote: Total 489 (delta 187), reused 246 (delta 127), pack-reused 156 (from 1)
Receiving objects: 100% (489/489), 2.60 MiB | 23.77 MiB/s, done.
Resolving deltas: 100% (256/256), done.
/content/silent_speech
Submodule 'text_alignments' (https://github.com/dgaddy/silent_speech_alignments.git) registered for path 'text_alignments'
Cloning into '/content/silent_speech/text_alignments'...
Submodule path 'text_alignments': checked out '5c71ae9fcbb94e74e19eb9547c3b404baf6126a7'


In [5]:
!pip install -q \
  speechbrain==1.0.3 timm jiwer==2.2.1 praat-textgrids \
  noisereduce resampy unidecode h5py soundfile librosa \
  transformers torchinfo torchprofile wandb omegaconf

# CTC decoder backend (the fragile bit) — flashlight-text + KenLM
!pip install -q flashlight-text
!pip install -q https://github.com/kpu/kenlm/archive/master.zip

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 9.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 864.1/864.1 kB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 124.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 133.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 134.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 67.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 23.2 MB/s eta 0:00:

In [6]:
%env DATA_PATH=/content/data
!mkdir -p /content/data

env: DATA_PATH=/content/data


In [7]:
!python download_data.py

2026-08-24 03:20:45.114283: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-24 03:20:45.131548: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787541645.152803    2066 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787541645.159295    2066 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-24 03:20:45.180374: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [10]:
!python data_collection/clean_audio.py

Streaming output truncated to the last 5000 lines.
cleaned /content/data/Gaddy/emg_data/nonparallel_data/4-23/374_audio.flac -> /content/data/Gaddy/emg_data/nonparallel_data/4-23/374_audio_resampled.flac
cleaned /content/data/Gaddy/emg_data/nonparallel_data/4-24/441_audio.flac -> /content/data/Gaddy/emg_data/nonparallel_data/4-24/441_audio_resampled.flac
cleaned /content/data/Gaddy/emg_data/nonparallel_data/4-26/413_audio.flac -> /content/data/Gaddy/emg_data/nonparallel_data/4-26/413_audio_resampled.flac
cleaned /content/data/Gaddy/emg_data/nonparallel_data/4-30/414_audio.flac -> /content/data/Gaddy/emg_data/nonparallel_data/4-30/414_audio_resampled.flac
cleaned /content/data/Gaddy/emg_data/nonparallel_data/4-21/351_audio.flac -> /content/data/Gaddy/emg_data/nonparallel_data/4-21/351_audio_resampled.flac
cleaned /content/data/Gaddy/emg_data/nonparallel_data/4-29/436_audio.flac -> /content/data/Gaddy/emg_data/nonparallel_data/4-29/436_audio_resampled.flac
cleaned /content/data/Gaddy/emg

In [11]:
!python build_hdf5.py

2026-08-24 03:42:46.830678: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-24 03:42:46.847907: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787542966.868843    7908 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787542966.875259    7908 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-24 03:42:46.896247: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [12]:
!cp /content/data/Gaddy/h5/emg_dataset.h5   /content/drive/MyDrive/silent_speech/
!cp /content/data/Gaddy/emg_data.tar.gz     /content/drive/MyDrive/silent_speech/

In [13]:
!python get_lexicon.py
!ls -la KenLM

2026-08-24 03:45:58.793586: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-24 03:45:58.810629: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787543158.831665    8848 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787543158.837989    8848 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-24 03:45:58.858708: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [23]:
# # only if KenLM/lm.bin is missing
!wget -q -O KenLM/lm.bin https://download.pytorch.org/torchaudio/decoder-assets/librispeech-4-gram/lm.bin

In [24]:
# keep a copy so later sessions skip this step
!cp -r KenLM /content/drive/MyDrive/silent_speech/

In [25]:
import json
cfg_path = "config/recognition_model.json"
cfg = json.load(open(cfg_path))
cfg["num_epochs"]     = 2                                             # tiny test
cfg["eval_interval"]  = 1                                             # check WER every epoch
cfg["num_workers"]    = 2                                             # Colab-safe
cfg["ckpt_directory"] = "/content/drive/MyDrive/silent_speech/output" # save to Drive
json.dump(cfg, open(cfg_path, "w"), indent=4)
print("config updated")

config updated


In [20]:
# delete the stray argument that breaks model construction
!sed -i '/norm_layer=norm_layer,/d' architecture.py

In [26]:
!python recognition_model.py

2026-08-24 03:59:50.942380: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-24 03:59:50.959645: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787543990.980847   12568 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787543990.987146   12568 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-24 03:59:51.007876: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

In [27]:
# FULL TRAIN
import json
cfg_path = "config/recognition_model.json"
cfg = json.load(open(cfg_path))
cfg["num_epochs"]    = 200   # repo default; lower to ~100 if short on time
cfg["eval_interval"] = 5
json.dump(cfg, open(cfg_path, "w"), indent=4)
print("ready for full run")

ready for full run


In [ ]:
!python recognition_model.py

2026-08-24 04:06:56.005869: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-24 04:06:56.023490: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1787544416.044885   14461 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1787544416.051265   14461 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-24 04:06:56.072236: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr